# Strings & String Algorithms: Zero to Hero

An array of characters with immutability bolted on — which changes the cost of everything, and
introduces the only topic in this series where the *definition* of "length" is contested.

> **Prerequisites:** [`arrays_zero_to_hero.ipynb`](arrays_zero_to_hero.ipynb) — a string is an
> array, and §2.4's quadratic accident has its most famous form here — and
> [`complexity_zero_to_hero.ipynb`](complexity_zero_to_hero.ipynb) for the measurement harness.

***

## Why this notebook is different

Two things usually taught as folklore, both measured here and both more interesting than the
folklore version:

- **The concatenation trap is real, CPython hides it, and the hiding place has two exits.**
  `s += chunk` in a loop is famously O(n²) — except §1.2 measures it as **exactly $O(n)$** while
  the result stays under about a megabyte, because CPython resizes the string in place when
  nothing else refers to it. Then the same function, on the same loop, measures as **exactly
  $O(n^2)$** once the result reaches a few megabytes and `realloc` can no longer extend in place.
  And holding **one extra reference** brings the quadratic back at any size — around 4 seconds
  against 0.02 at n = 40,000. Two independent cliffs under one optimisation nobody promised you.
- **Naive matching survives because the worst case is hard to reach.** §3 measures it at **1.00
  comparisons per character** on English-like text and **498 per character** on a constructed
  adversarial input. Same algorithm, same code, a 500× difference decided entirely by the data.

And one thing that is not folklore at all: **`len()` does not tell you how many characters there
are.** §1.4 measures a family emoji whose Python `len()` is 5, whose UTF-8 length is 18 bytes, and
which Java reports as 8. Every one of those numbers is correct.

| Part | What it covers |
|---|---|
| **0. Setup** | Imports, JDK check, `dsa_toolkit` |
| **1. Theory from zero** | Strings as immutable arrays · **the concatenation trap and the optimisation that hides it** · builders · **encodings, and why `len()` surprises you** |
| **2. Searching** | Naive matching · **KMP with the prefix function derived** · Z-algorithm · **Rabin-Karp and a real hash collision** |
| **3. The signature difficulty** | **Why naive matching survives** — the gap between worst case and typical, measured |
| **4. Tough questions** | 12 questions + 3 coding challenges |
| **5. Practice** | 8 exercises |
| **6. Reading** | The papers behind each section |
| **Appendix** | String-specific errors and a checklist |

## The one-paragraph summary

A string is a **contiguous array of code units** that you are not allowed to modify, and almost
everything follows from the second half of that sentence: every "modification" allocates a new
string and copies, so building one incrementally is quadratic unless you use a **builder** — or
unless your runtime is quietly optimising behind your back, which CPython is and which makes the
bug appear only sometimes. Searching within a string is where the interesting algorithms live:
**naive** matching is O(nm) in theory and about one comparison per character on real text, **KMP** and **Z** are O(n+m)
by never re-examining a character, and **Rabin-Karp** trades exactness for a rolling hash and
therefore needs a verification step that people forget. Underneath all of it, "character" is not
well defined — bytes, code units, code points and grapheme clusters are four different things, and
picking the wrong one is how you get a bug that only appears for users whose names you did not
test with.

***
# Part 0 - Setup

In [1]:
# ---------------------------------------------------------------------------
# Everything this notebook uses. Standard library only.
# ---------------------------------------------------------------------------
import gc
import random
import sys
import time
import unicodedata

from dsa_toolkit import (JavaError, StressFailure, cross_check, growth_table,
                         java_available, measure_growth, run_java, stress)

RANDOM_SEED = 12345

ok, detail = java_available()
print("python", sys.version.split()[0])
print("JDK available:", ok, "|", detail)

python 3.14.7
JDK available: True | javac 25.0.4.1


***
# Part 1 - Theory from zero

1. A string is an immutable array
2. **The concatenation trap, and the optimisation that hides it**
3. Builders, and what they actually do
4. **Encodings: bytes, code units, code points, graphemes**

## 1.1 A string is an immutable array

Everything NB-01 §1.1 said about arrays applies: contiguous storage, O(1) indexing by address
arithmetic, cheap sequential traversal. Then one extra rule is added, and it changes every cost in
the table: **you cannot modify a string in place.**

Immutability is not an oversight. It buys:

- **Safe sharing.** A string can be passed around, used as a dictionary key, and cached without
  defensive copying, because nobody can change it underneath you.
- **A cacheable hash.** Python computes a string's hash once and stores it. NB-03 depends on this.
- **Interning.** Identical literals can share one object (Java does this eagerly for literals).

And it costs exactly one thing, from which every performance problem in this notebook follows:
**every "modification" allocates a new string and copies.**

| Operation | Cost | Because |
|---|---|---|
| `s[i]` | $\Theta(1)$ | array indexing |
| `len(s)` | $\Theta(1)$ | stored, not counted |
| `s + t` | $\Theta(n+m)$ | allocates and copies **both** |
| `s.upper()`, `s.replace(...)` | $\Theta(n)$ | builds a new string |
| `s[i:j]` | $\Theta(j-i)$ | copies the slice |
| `sub in s` | $\Theta(nm)$ worst case | §2 and §3 |
| `"".join(parts)` | $\Theta(\text{total})$ | one allocation, one pass |

In [2]:
# ---------------------------------------------------------------------------
# Immutability, demonstrated rather than asserted.
# ---------------------------------------------------------------------------
s = "hello"
print("s        =", repr(s), "  id =", id(s))
try:
    s[0] = "H"
except TypeError as exc:
    print("s[0]='H' ->", type(exc).__name__ + ":", exc)

t = s.upper()
print("s.upper() =", repr(t), "  id =", id(t), " <- a different object")
print("s is unchanged:", repr(s))
print()

# Every 'modification' is an allocation. Watch the identity change.
acc = ""
ids = []
for ch in "abcd":
    acc += ch
    ids.append(id(acc))
print("building 'abcd' one character at a time produced these object ids:")
print(" ", ids)
print("  distinct objects:", len(set(ids)))
print()
print("That is the whole story of this notebook. Each += made a new string, and")
print("a new string means copying everything already there. Section 1.2 measures")
print("what that costs -- and finds it is not what you would expect.")

s        = 'hello'   id = 2389738570544
s[0]='H' -> TypeError: 'str' object does not support item assignment
s.upper() = 'HELLO'   id = 2389756405840  <- a different object
s is unchanged: 'hello'

building 'abcd' one character at a time produced these object ids:
  [140733070043384, 2389761292256, 2389761308256, 2389761292256]
  distinct objects: 3

That is the whole story of this notebook. Each += made a new string, and
a new string means copying everything already there. Section 1.2 measures
what that costs -- and finds it is not what you would expect.


## 1.2 The concatenation trap, and the optimisation that hides it

The textbook claim: building a string with `s += chunk` in a loop is $\Theta(n^2)$, because
iteration $i$ copies the $i$ characters accumulated so far, and $\sum i = n^2/2$.

The claim is correct about the algorithm. **It is often wrong about CPython**, and the reason is
worth knowing, because it turns a predictable cost into a cliff.

CPython's `+=` on strings checks whether the left operand's reference count is 1 — whether *this
variable is the only thing pointing at that string*. If so, nothing else can observe the change,
so it resizes the buffer in place instead of allocating a copy. The quadratic behaviour vanishes.

Until something else holds a reference. Then it cannot, and the cost reappears in full.

We are going to measure this on **two** axes, because the optimisation fails on both — and the
second one is not the one people warn you about.

In [3]:
# ---------------------------------------------------------------------------
# Three ways to build the same string.
# ---------------------------------------------------------------------------
CHUNK = "abcdefghij"


def build_plus_equals(n):
    """The 'quadratic' loop -- but nothing else references s."""
    s = ""
    for _ in range(n):
        s += CHUNK
    return s


def build_join(n):
    """The recommended way: collect, then join once."""
    parts = []
    for _ in range(n):
        parts.append(CHUNK)
    return "".join(parts)


def build_plus_equals_aliased(n):
    """Identical to the first, except one extra reference is kept alive."""
    s = ""
    keep = []
    for _ in range(n):
        keep.append(s)          # now s has refcount > 1, every iteration
        s += CHUNK
    return s


assert build_plus_equals(50) == build_join(50) == build_plus_equals_aliased(50)
print("all three produce identical strings.\n")

# Warm up and settle the heap first: the cells above allocated tens of MB, and
# the first timing in a cell otherwise pays for someone else's garbage.
gc.collect()
build_plus_equals(20_000)

print("s += chunk, nothing else referencing s, result 50 KB -> 400 KB:")
growth_table(measure_growth(build_plus_equals, [5_000, 10_000, 20_000, 40_000],
                            repeats=7), claim="O(n)")

all three produce identical strings.

s += chunk, nothing else referencing s, result 50 KB -> 400 KB:
         n        seconds      ratio
------------------------------------
     5,000       0.000845          -
    10,000       0.001620       1.92
    20,000       0.003222       1.99
    40,000       0.006457       2.00

best fit: O(n) (relative error 0.020); next: O(n log n) (0.098)
claimed O(n) -> measurement MATCHES the claim


[('O(n)', 0.019530484471780282),
 ('O(n log n)', 0.09799959604229058),
 ('O(log n)', 1.1150706915740844),
 ('O(1)', 1.3939460127628103),
 ('O(2^n)', 1.3939460127628103),
 ('O(n^2)', 1.5209270791327627),
 ('O(n^3)', 10.751888483040542)]

Linear to within about a percent — ratios of 2.0 per doubling, and a fit that leaves the
$O(n \log n)$ candidate an order of magnitude behind. The textbook's quadratic is nowhere to be
seen.

Those sizes were not chosen innocently. They stop at a 400 KB result, and the reason is the next
cell: push the same table up to a result of 800 KB and its last ratio starts drifting towards 2.5
rather than 2.0. Now keep the code identical and just **make the strings bigger**.

In [4]:
# Identical function, identical loop, nothing else referencing s.
# The only change is n -- the result grows from 2 MB to 16 MB.
gc.collect()
print("s += chunk, nothing else referencing s, result 2 MB -> 16 MB:")
growth_table(measure_growth(build_plus_equals,
                            [200_000, 400_000, 800_000, 1_600_000], repeats=3),
             claim="O(n^2)")

s += chunk, nothing else referencing s, result 2 MB -> 16 MB:


         n        seconds      ratio
------------------------------------
   200,000       0.115714          -
   400,000       0.478053       4.13
   800,000       1.806634       3.78
 1,600,000       6.813752       3.77

best fit: O(n^2) (relative error 0.043); next: O(n log n) (1.165)
claimed O(n^2) -> measurement MATCHES the claim


[('O(n^2)', 0.04278596168751877),
 ('O(n log n)', 1.165449897118973),
 ('O(n)', 1.3672901316430897),
 ('O(n^3)', 1.6283386560191149),
 ('O(log n)', 8.340807605316396),
 ('O(1)', 9.651111537699677),
 ('O(2^n)', 9.651111537699677)]

**The same code is $\Theta(n)$ at one size and $\Theta(n^2)$ at another**, with a visible
transition in between — which is why the linear table above stops where it does.

The in-place resize is a `realloc`. `realloc` can only extend a block without copying if the memory
immediately after it happens to be free — and once the block is a few megabytes, it is served
directly by the operating system's allocator, where that stops being true. So every iteration
copies, and the quadratic returns on its own, with no second reference anywhere in sight.

That is the first axis. The second is the one people do warn you about: hold **one** extra
reference and the optimisation is disabled outright, at any size.

Note also what is being timed in the next cell. The claim "`''.join(parts)` is $O(n)$" is about the
**join**, so that is what gets measured — the list is built in `setup`, whose cost `measure_growth`
excludes. Time the loop that fills the list along with it and you are measuring the loop instead,
which drags the fit to `O(n log n)` and says nothing about `join`. Measure the thing you are
claiming about.

In [5]:
gc.collect()
"".join([CHUNK] * 1_600_000)      # warm up at the largest size
print("''.join(parts), timing the join alone:")
growth_table(measure_growth(lambda parts: "".join(parts),
                            [200_000, 400_000, 800_000, 1_600_000],
                            setup=lambda n: [CHUNK] * n, repeats=7),
             claim="O(n)")
print()
print("s += chunk, with ONE extra reference held:")
growth_table(measure_growth(build_plus_equals_aliased,
                            [5_000, 10_000, 20_000, 40_000], repeats=3),
             claim="O(n^2)")

''.join(parts), timing the join alone:


         n        seconds      ratio
------------------------------------
   200,000       0.001126          -
   400,000       0.002308       2.05
   800,000       0.004642       2.01
 1,600,000       0.009376       2.02

best fit: O(n) (relative error 0.015); next: O(n log n) (0.045)
claimed O(n) -> measurement MATCHES the claim

s += chunk, with ONE extra reference held:


         n        seconds      ratio
------------------------------------
     5,000       0.073778          -
    10,000       0.287027       3.89
    20,000       1.079848       3.76
    40,000       4.204886       3.89

best fit: O(n^2) (relative error 0.046); next: O(n log n) (1.038)
claimed O(n^2) -> measurement MATCHES the claim


[('O(n^2)', 0.04648491804662369),
 ('O(n log n)', 1.0382099705326715),
 ('O(n)', 1.3011533624041913),
 ('O(n^3)', 1.6516778769950045),
 ('O(log n)', 7.57338819064684),
 ('O(1)', 9.281462255885073),
 ('O(2^n)', 9.281462255885073)]

In [6]:
# ---------------------------------------------------------------------------
# The cliff, in absolute terms.
# ---------------------------------------------------------------------------
gc.collect()
for _f in (build_plus_equals, build_join, build_plus_equals_aliased):
    _f(2_000)                    # warm up; the table below starts at 10,000
print("Same n, same output, three implementations:\n")
print("  %10s %14s %14s %18s" % ("n", "+= (s)", "join (s)", "+= aliased (s)"))
print("  " + "-" * 62)
for n in [10_000, 20_000, 40_000]:
    t1 = measure_growth(build_plus_equals, [n], repeats=3)[0]["seconds"]
    t2 = measure_growth(build_join, [n], repeats=3)[0]["seconds"]
    t3 = measure_growth(build_plus_equals_aliased, [n], repeats=3)[0]["seconds"]
    print("  %10s %14.5f %14.5f %18.5f" % ("{:,}".format(n), t1, t2, t3))

print()
print("Read the last two columns against each other. The ONLY difference between")
print("them is one `keep.append(s)` -- a line that does not touch the string being")
print("built. It costs two orders of magnitude.")
print()
print("Three things to take from this:")
print("  1. The textbook O(n^2) claim is right about the ALGORITHM. CPython's")
print("     in-place resize is an implementation detail that happens to rescue it.")
print("  2. Relying on that detail is a trap, because it disappears silently. A")
print("     debugger, a logging call, a list of intermediate values, a closure --")
print("     anything that takes a second reference -- brings the quadratic back.")
print("  3. So write ''.join(parts). It is O(n) unconditionally, it is faster than")
print("     the optimised += anyway, and it cannot regress.")
print()
print("The same reasoning applies to bytes: use bytearray, which is genuinely")
print("mutable, rather than relying on b'' += b'' being optimised.")

Same n, same output, three implementations:

           n         += (s)       join (s)     += aliased (s)
  --------------------------------------------------------------


      10,000        0.00151        0.00055            0.22402


      20,000        0.00800        0.00087            1.10326


      40,000        0.01789        0.00209            4.34519

Read the last two columns against each other. The ONLY difference between
them is one `keep.append(s)` -- a line that does not touch the string being
built. It costs two orders of magnitude.

Three things to take from this:
  1. The textbook O(n^2) claim is right about the ALGORITHM. CPython's
     in-place resize is an implementation detail that happens to rescue it.
  2. Relying on that detail is a trap, because it disappears silently. A
     debugger, a logging call, a list of intermediate values, a closure --
     anything that takes a second reference -- brings the quadratic back.
  3. So write ''.join(parts). It is O(n) unconditionally, it is faster than
     the optimised += anyway, and it cannot regress.

The same reasoning applies to bytes: use bytearray, which is genuinely
mutable, rather than relying on b'' += b'' being optimised.


## 1.3 Builders

Every language provides the same escape hatch under a different name: accumulate the pieces in a
mutable structure, and materialise the string once.

| Language | Builder | Note |
|---|---|---|
| Python | `list` + `"".join(...)` | there is no `StringBuilder`; the list *is* the builder |
| Python (bytes) | `bytearray` | genuinely mutable, unlike `bytes` |
| Java | `StringBuilder` | not thread-safe, and that is why it is the fast one |
| Java | `StringBuffer` | synchronised; you almost never want it |

Java's case is starker than Python's, because the JVM has no equivalent in-place optimisation for
`String +=` inside a loop. The compiler rewrites `a + b` into a `StringBuilder` for a *single*
expression, but each loop iteration gets its own builder, so the copying is unavoidable.

In [7]:
# ---------------------------------------------------------------------------
# Java: String += vs StringBuilder. No hidden optimisation to rescue it.
# ---------------------------------------------------------------------------
JAVA_CONCAT = """
public class Concat {
    public static void main(String[] args) {
        int n = Integer.parseInt(args[0]);

        long t0 = System.nanoTime();
        String s = "";
        for (int i = 0; i < n; i++) s += "abcdefghij";   // new String every iteration
        double plus = (System.nanoTime() - t0) / 1e9;

        t0 = System.nanoTime();
        StringBuilder sb = new StringBuilder();
        for (int i = 0; i < n; i++) sb.append("abcdefghij");
        String t = sb.toString();
        double builder = (System.nanoTime() - t0) / 1e9;

        System.out.printf("%.6f %.6f %b%n", plus, builder, s.equals(t));
    }
}
"""

print("  %10s %16s %20s %12s" % ("n", "String += (s)", "StringBuilder (s)", "ratio"))
print("  " + "-" * 64)
for n in [10_000, 20_000, 40_000]:
    out = run_java(JAVA_CONCAT, args=[n]).split()
    plus, builder = float(out[0]), float(out[1])
    print("  %10s %16.5f %20.6f %11.0fx   (identical: %s)"
          % ("{:,}".format(n), plus, builder, plus / builder, out[2]))

print()
print("The ratio GROWS with n, which is the signature of a complexity difference")
print("rather than a constant factor: O(n^2) against O(n).")
print()
print("Note this is Java's honest behaviour, with no equivalent of CPython's")
print("in-place resize. Which of the two languages is more helpful here is")
print("debatable -- Java is slower and predictable, CPython is faster and has a")
print("cliff. Predictable is easier to reason about.")

           n    String += (s)    StringBuilder (s)        ratio
  ----------------------------------------------------------------


      10,000          0.10730             0.001428          75x   (identical: true)


      20,000          0.32850             0.002047         160x   (identical: true)


      40,000          1.08051             0.003305         327x   (identical: true)

The ratio GROWS with n, which is the signature of a complexity difference
rather than a constant factor: O(n^2) against O(n).

Note this is Java's honest behaviour, with no equivalent of CPython's
in-place resize. Which of the two languages is more helpful here is
debatable -- Java is slower and predictable, CPython is faster and has a
cliff. Predictable is easier to reason about.


## 1.4 Encodings: bytes, code units, code points, graphemes

Here is the question with four correct answers: **how long is a string?**

Four different things get called "a character", and every language picks one to expose as
`len()`:

| Term | What it is | Python `len()` counts | Java `.length()` counts |
|---|---|---|---|
| **byte** | one octet of an encoding | no | no |
| **code unit** | the fixed-size piece an encoding works in — 1 byte for UTF-8, **2 for UTF-16** | no | **yes** |
| **code point** | one entry in the Unicode table, `U+0041` | **yes** | no |
| **grapheme cluster** | what a human calls a character | no | no |

**Neither language counts what a user would call a character.** Python counts code points, so a
combining accent counts twice. Java counts UTF-16 code units, so anything outside the Basic
Multilingual Plane — emoji, historic scripts, many CJK extensions — counts twice.

In [8]:
# ---------------------------------------------------------------------------
# The same strings, measured four ways.
# ---------------------------------------------------------------------------
SAMPLES = [
    ("plain ascii", "cafe"),
    ("precomposed e-acute", "café"),
    ("e + combining acute", "café"),
    ("musical G clef U+1D11E", "\U0001D11E"),
    ("family emoji (ZWJ)", "\U0001F468‍\U0001F469‍\U0001F467"),
    ("flag (2 regional ind.)", "\U0001F1EE\U0001F1F3"),
]

print("  %-26s %10s %10s %14s" % ("string", "len()", "utf-8 B", "utf-16 units"))
print("  " + "-" * 64)
for label, s in SAMPLES:
    print("  %-26s %10d %10d %14d"
          % (label, len(s), len(s.encode("utf-8")), len(s.encode("utf-16-le")) // 2))

print()
print("Every column is correct. They answer different questions.")
print()
print("  'len()'       -> code points. Python's answer.")
print("  'utf-16 units'-> what Java's String.length() returns.")
print("  'utf-8 B'     -> what your database column limit and your network")
print("                   payload are actually measured in.")
print()
print("The family emoji is the clearest case: one thing on screen, 5 code points,")
print("8 UTF-16 units, 18 bytes. A VARCHAR(10) will reject it.")

  string                          len()    utf-8 B   utf-16 units
  ----------------------------------------------------------------
  plain ascii                         4          4              4
  precomposed e-acute                 4          5              4
  e + combining acute                 5          6              5
  musical G clef U+1D11E              1          4              2
  family emoji (ZWJ)                  5         18              8
  flag (2 regional ind.)              2          8              4

Every column is correct. They answer different questions.

  'len()'       -> code points. Python's answer.
  'utf-16 units'-> what Java's String.length() returns.
  'utf-8 B'     -> what your database column limit and your network
                   payload are actually measured in.

The family emoji is the clearest case: one thing on screen, 5 code points,
8 UTF-16 units, 18 bytes. A VARCHAR(10) will reject it.


In [9]:
# ---------------------------------------------------------------------------
# The equality bug this causes, and the fix.
# ---------------------------------------------------------------------------
precomposed = "café"          # U+00E9
decomposed = "café"          # 'e' + U+0301 combining acute

print("precomposed:", repr(precomposed), "len", len(precomposed))
print("decomposed :", repr(decomposed), "len", len(decomposed))
print("they print identically:      %s  vs  %s" % (precomposed, decomposed))
print("precomposed == decomposed :", precomposed == decomposed)
print()
print("NORMALISE before comparing:")
for form in ("NFC", "NFD"):
    a = unicodedata.normalize(form, precomposed)
    b = unicodedata.normalize(form, decomposed)
    print("  %s: equal=%s   len %d vs %d" % (form, a == b, len(a), len(b)))

print()
print("This is a real class of bug: a username, filename or password that looks")
print("right and does not match. macOS stores filenames as NFD and most other")
print("systems use NFC, so the same name round-tripped through both compares")
print("unequal.")
print()
print("The rule: normalise (usually NFC) at the boundary where text enters your")
print("system, once, and compare normalised forms thereafter.")

precomposed: 'café' len 4
decomposed : 'café' len 5
they print identically:      café  vs  café
precomposed == decomposed : False

NORMALISE before comparing:
  NFC: equal=True   len 4 vs 4
  NFD: equal=True   len 5 vs 5

This is a real class of bug: a username, filename or password that looks
right and does not match. macOS stores filenames as NFD and most other
systems use NFC, so the same name round-tripped through both compares
unequal.

The rule: normalise (usually NFC) at the boundary where text enters your
system, once, and compare normalised forms thereafter.


In [10]:
# ---------------------------------------------------------------------------
# Java's view of the same strings, and the surrogate pair problem.
# ---------------------------------------------------------------------------
JAVA_UNICODE = """
public class Uni {
    public static void main(String[] args) {
        String clef  = "\\uD834\\uDD1E";          // U+1D11E as a surrogate pair
        String cafe1 = "caf\\u00e9";
        String cafe2 = "cafe\\u0301";

        System.out.println("G clef  : length()=" + clef.length()
                           + "  codePointCount=" + clef.codePointCount(0, clef.length()));
        System.out.println("        : charAt(0) alone is a lone surrogate, not a character");
        System.out.println("cafe1   : length()=" + cafe1.length());
        System.out.println("cafe2   : length()=" + cafe2.length()
                           + "  cafe1.equals(cafe2)=" + cafe1.equals(cafe2));

        String a = "hello", b = "hello";          // interned literals
        String c = new String("hello");           // a fresh object
        String d = c.intern();
        System.out.println("interning: a==b " + (a == b) + ", a==c " + (a == c)
                           + ", a==c.intern() " + (a == d)
                           + ", a.equals(c) " + a.equals(c));
    }
}
"""
print(run_java(JAVA_UNICODE))
print("Two Java-specific traps in that output:")
print()
print("  length()==2 for ONE character. Java strings are UTF-16, so any code")
print("  point above U+FFFF occupies two chars. Iterating with charAt() splits")
print("  emoji in half; use codePoints() instead.")
print()
print("  a==b is TRUE for two separate literals, because the compiler interns")
print("  them into one object -- and a==c is FALSE for an identical string built")
print("  at runtime. This is why == on strings is a bug that passes its tests:")
print("  it works for literals and fails for anything read from input.")
print("  Always .equals().")

G clef  : length()=2  codePointCount=1
        : charAt(0) alone is a lone surrogate, not a character
cafe1   : length()=4
cafe2   : length()=5  cafe1.equals(cafe2)=false
interning: a==b true, a==c false, a==c.intern() true, a.equals(c) true

Two Java-specific traps in that output:

  length()==2 for ONE character. Java strings are UTF-16, so any code
  point above U+FFFF occupies two chars. Iterating with charAt() splits
  emoji in half; use codePoints() instead.

  a==b is TRUE for two separate literals, because the compiler interns
  them into one object -- and a==c is FALSE for an identical string built
  at runtime. This is why == on strings is a bug that passes its tests:
  it works for literals and fails for anything read from input.
  Always .equals().


***
# Part 2 - Searching inside a string

Find every position where a pattern of length $m$ occurs in a text of length $n$. Four algorithms,
each built from the previous one's weakness.

| Algorithm | Worst case | Preprocessing | Idea |
|---|---|---|---|
| **naive** | $\Theta(nm)$ | none | try every alignment |
| **KMP** | $\Theta(n+m)$ | $\Theta(m)$ | on a mismatch, reuse what already matched |
| **Z-algorithm** | $\Theta(n+m)$ | $\Theta(n+m)$ | same information, computed differently |
| **Rabin-Karp** | $\Theta(nm)$ worst, $\Theta(n+m)$ expected | $\Theta(m)$ | compare hashes, verify hits |

Every implementation below is checked against a brute-force reference over thousands of generated
inputs, including the empty pattern and the empty text — which is where these functions actually
break.

## 2.1 Naive matching

Try every starting position; at each, compare until a mismatch. $n-m+1$ alignments, up to $m$
comparisons each, so $\Theta(nm)$ worst case.

Its reputation is worse than it deserves, and §3 is about why. First, a correct implementation to
measure against.

In [11]:
# ---------------------------------------------------------------------------
# Naive search, instrumented so we can count comparisons rather than just time.
# ---------------------------------------------------------------------------
def naive_search_all(case):
    """Every start index where pat occurs in text."""
    text, pat = case
    n, m = len(text), len(pat)
    if m == 0:
        return list(range(n + 1))          # the empty pattern matches everywhere
    out = []
    for i in range(n - m + 1):
        j = 0
        while j < m and text[i + j] == pat[j]:
            j += 1
        if j == m:
            out.append(i)
    return out


def reference_search_all(case):
    """Slicing reference. Obviously correct, obviously slow."""
    text, pat = case
    n, m = len(text), len(pat)
    if m == 0:
        return list(range(n + 1))
    return [i for i in range(n - m + 1) if text[i:i + m] == pat]


def gen_case(r):
    text = "".join(r.choice("aab") for _ in range(r.randrange(0, 40)))
    pat = "".join(r.choice("aab") for _ in range(r.randrange(0, 5)))
    return (text, pat)


EDGE = [("", ""), ("", "a"), ("a", ""), ("a", "a"), ("aaa", "a"), ("aaaa", "aa"),
        ("abababab", "abab"), ("aaaaab", "aab"), ("abc", "abcd")]

n = stress(naive_search_all, reference_search_all, gen_case, n=5000, extra=EDGE,
           label="naive_search_all")
print("naive: %s cases agree with the reference, including every empty-input case."
      % "{:,}".format(n))
print()
print("  'abracadabra', pattern 'abra' ->", naive_search_all(("abracadabra", "abra")))
print("  'aaaa', pattern 'aa'          ->", naive_search_all(("aaaa", "aa")),
      "  <- overlapping matches count")
print("  'abc', pattern ''             ->", naive_search_all(("abc", "")),
      "  <- the empty pattern is everywhere")

naive: 5,009 cases agree with the reference, including every empty-input case.



  'abracadabra', pattern 'abra' -> [0, 7]
  'aaaa', pattern 'aa'          -> [0, 1, 2]   <- overlapping matches count
  'abc', pattern ''             -> [0, 1, 2, 3]   <- the empty pattern is everywhere


## 2.2 KMP: never re-examine a character

Naive matching throws information away. If the pattern `abcabd` matched five characters and failed
on the sixth, we *know* the last three characters read were `abc` — and `abc` is a prefix of the
pattern. Sliding back to re-read them is wasted work.

**The prefix function** captures exactly that. For pattern $p$, define

$$ \pi[i] = \text{the length of the longest \emph{proper} prefix of } p[0..i] \text{ that is also a suffix of it.} $$

"Proper" means shorter than the whole thing, or the answer would always be $i+1$.

For `abacabab`: $\pi = [0,0,1,0,1,2,3,2]$. At index 6 the prefix `abacaba` has `aba` as both a
prefix and a suffix, so $\pi[6]=3$.

**Why it makes search linear.** On a mismatch at pattern position $k$, instead of restarting we
set $k \leftarrow \pi[k-1]$ — the next-longest prefix that could still match — and the text pointer
never moves backwards. Each character is examined once going forward, and $k$ only decreases,
so the total work is $\Theta(n+m)$ by the same amortised argument as NB-01 §2.3's sliding window.

In [12]:
# ---------------------------------------------------------------------------
# The prefix function, and a brute-force reference to check it against.
# ---------------------------------------------------------------------------
def prefix_function(p):
    """pi[i] = length of the longest proper prefix of p[:i+1] that is also a suffix."""
    pi = [0] * len(p)
    k = 0                                  # length of the current candidate border
    for i in range(1, len(p)):
        while k and p[i] != p[k]:
            k = pi[k - 1]                  # fall back to the next-longest border
        if p[i] == p[k]:
            k += 1
        pi[i] = k
    return pi


def prefix_function_naive(p):
    """Check every candidate length directly. O(n^3), and obviously right."""
    out = []
    for i in range(1, len(p) + 1):
        s, best = p[:i], 0
        for L in range(1, i):
            if s[:L] == s[i - L:]:
                best = L
        out.append(best)
    return out


print("  %-12s %s" % ("pattern", "prefix function"))
print("  " + "-" * 44)
for w in ["abacabab", "aabaaab", "abab", "aaaa", "abcde"]:
    print("  %-12s %s" % (repr(w), prefix_function(w)))

n = stress(prefix_function, prefix_function_naive,
           lambda r: "".join(r.choice("ab") for _ in range(r.randrange(0, 18))),
           n=4000, extra=["", "a", "aa", "ab", "aaaa", "abab", "aabaa"],
           label="prefix_function")
print("\n%s random patterns: identical to the brute-force definition."
      % "{:,}".format(n))

  pattern      prefix function
  --------------------------------------------
  'abacabab'   [0, 0, 1, 0, 1, 2, 3, 2]
  'aabaaab'    [0, 1, 0, 1, 2, 2, 3]
  'abab'       [0, 0, 1, 2]
  'aaaa'       [0, 1, 2, 3]
  'abcde'      [0, 0, 0, 0, 0]

4,007 random patterns: identical to the brute-force definition.


In [13]:
# ---------------------------------------------------------------------------
# KMP search, built on the prefix function.
# ---------------------------------------------------------------------------
def kmp_search_all(case):
    text, pat = case
    if not pat:
        return list(range(len(text) + 1))
    pi = prefix_function(pat)
    out, k = [], 0
    for i, ch in enumerate(text):          # i never goes backwards
        while k and ch != pat[k]:
            k = pi[k - 1]
        if ch == pat[k]:
            k += 1
        if k == len(pat):
            out.append(i - len(pat) + 1)
            k = pi[k - 1]                  # keep going, to find overlaps
    return out


n = stress(kmp_search_all, reference_search_all, gen_case, n=6000, extra=EDGE,
           label="kmp_search_all")
print("KMP: %s cases identical to the reference." % "{:,}".format(n))
print()
print("  'aaaa', pattern 'aa' ->", kmp_search_all(("aaaa", "aa")),
      "  <- overlaps preserved")
print()
print("The `k = pi[k - 1]` after a full match is what makes overlapping matches")
print("work. Setting k = 0 there would find only non-overlapping occurrences --")
print("a different (and sometimes wanted) definition worth being explicit about.")

KMP: 6,009 cases identical to the reference.

  'aaaa', pattern 'aa' -> [0, 1, 2]   <- overlaps preserved

The `k = pi[k - 1]` after a full match is what makes overlapping matches
work. Setting k = 0 there would find only non-overlapping occurrences --
a different (and sometimes wanted) definition worth being explicit about.


## 2.3 The Z-algorithm

The same linear-time result reached from a different direction. For a string $s$, define

$$ z[i] = \text{the length of the longest substring starting at } i \text{ that is also a prefix of } s. $$

The algorithm maintains the rightmost interval $[lo, hi)$ known to match a prefix — the "Z-box" —
and reuses it: if $i$ falls inside the box, the answer at the corresponding position nearer the
front is already known, and only the part beyond $hi$ needs fresh comparisons. `hi` never
decreases, so those fresh comparisons total $O(n)$ across the whole run.

**For searching**, run it on `pattern + separator + text`, where the separator appears in neither.
Any position where $z[i] = |pattern|$ is a match.

Z is often easier to get right than KMP, and it is the natural tool when you want prefix-match
lengths at *every* position rather than just the match positions.

In [14]:
# ---------------------------------------------------------------------------
# Z-function, with the box logic spelled out.
# ---------------------------------------------------------------------------
def z_function(s):
    n = len(s)
    z = [0] * n
    if n:
        z[0] = n                           # by convention
    lo = hi = 0                            # the current Z-box [lo, hi)
    for i in range(1, n):
        if i < hi:                         # inside the box: reuse a known answer
            z[i] = min(hi - i, z[i - lo])
        while i + z[i] < n and s[z[i]] == s[i + z[i]]:
            z[i] += 1                      # extend past the box, one char at a time
        if i + z[i] > hi:                  # a new rightmost box
            lo, hi = i, i + z[i]
    return z


def z_naive(s):
    n = len(s)
    out = [0] * n
    if n:
        out[0] = n
    for i in range(1, n):
        k = 0
        while i + k < n and s[k] == s[i + k]:
            k += 1
        out[i] = k
    return out


print("  %-14s %s" % ("string", "z"))
print("  " + "-" * 46)
for w in ["abacaba", "aabxaayaab", "aaaa"]:
    print("  %-14s %s" % (repr(w), z_function(w)))

n = stress(z_function, z_naive,
           lambda r: "".join(r.choice("aab") for _ in range(r.randrange(0, 25))),
           n=5000, extra=["", "a", "aa", "aaaa", "abacaba", "aabaaab"],
           label="z_function")
print("\n%s random strings: identical to brute force." % "{:,}".format(n))
print()


def z_search_all(case):
    """Search by running Z over pattern + sep + text."""
    text, pat = case
    if not pat:
        return list(range(len(text) + 1))
    sep = "\x00"
    assert sep not in text and sep not in pat, "separator must not occur in the input"
    z = z_function(pat + sep + text)
    m = len(pat)
    return [i - m - 1 for i in range(m + 1, len(z)) if z[i] >= m]


n = stress(z_search_all, reference_search_all, gen_case, n=5000, extra=EDGE,
           label="z_search_all")
print("Z-based search: %s cases identical to the reference." % "{:,}".format(n))
print()
print("Worst case for Z is an all-equal string, where every box extends maximally:")
growth_table(measure_growth(z_function, [100_000, 200_000, 400_000, 800_000],
                            setup=lambda n_: "a" * n_, repeats=3), claim="O(n)")

  string         z
  ----------------------------------------------
  'abacaba'      [7, 0, 1, 0, 3, 0, 1]
  'aabxaayaab'   [10, 1, 0, 0, 2, 1, 0, 3, 1, 0]
  'aaaa'         [4, 3, 2, 1]

5,006 random strings: identical to brute force.

Z-based search: 5,009 cases identical to the reference.

Worst case for Z is an all-equal string, where every box extends maximally:


         n        seconds      ratio
------------------------------------
   100,000       0.047790          -
   200,000       0.100675       2.11
   400,000       0.205376       2.04
   800,000       0.406294       1.98

best fit: O(n) (relative error 0.028); next: O(n log n) (0.042)
claimed O(n) -> measurement MATCHES the claim


[('O(n)', 0.027919088599711477),
 ('O(n log n)', 0.042461111840284754),
 ('O(log n)', 1.3355362102654886),
 ('O(n^2)', 1.3979077021930522),
 ('O(1)', 1.5760627028461283),
 ('O(2^n)', 1.5760627028461283),
 ('O(n^3)', 9.854134461838903)]

## 2.4 Rabin-Karp, and the verification step people forget

A different trade: instead of comparing characters, compare a **hash** of the window against the
hash of the pattern. Comparing two integers is $O(1)$ regardless of $m$.

That only helps if the window's hash can be updated in $O(1)$ as the window slides, which is what a
**rolling hash** provides. Treating the window as a base-$b$ number modulo $M$:

$$ h_{i+1} = \big( (h_i - \text{text}[i] \cdot b^{m-1}) \cdot b + \text{text}[i+m] \big) \bmod M $$

Drop the leading character's contribution, shift, add the new one.

**And then verify.** Equal hashes do not mean equal strings. Skip the character-by-character check
on a hash hit and the algorithm returns matches that are not there — which the cell below
demonstrates by shrinking the modulus until it happens.

In [15]:
# ---------------------------------------------------------------------------
# Rabin-Karp, with the verification step made optional so we can watch it matter.
# ---------------------------------------------------------------------------
def rabin_karp_all(text, pat, base=256, mod=(1 << 61) - 1, verify=True):
    """Returns (matches, hash_hits). hash_hits - len(matches) = false positives."""
    n, m = len(text), len(pat)
    if m == 0:
        return list(range(n + 1)), 0
    if m > n:
        return [], 0
    high = pow(base, m - 1, mod)           # b^(m-1), for removing the leading char
    hp = ht = 0
    for i in range(m):                     # hash the pattern and the first window
        hp = (hp * base + ord(pat[i])) % mod
        ht = (ht * base + ord(text[i])) % mod

    out, hits = [], 0
    for i in range(n - m + 1):
        if hp == ht:
            hits += 1
            if not verify or text[i:i + m] == pat:
                out.append(i)
        if i + m < n:                      # roll the window forward
            ht = ((ht - ord(text[i]) * high) * base + ord(text[i + m])) % mod
    return out, hits


n = stress(lambda c: rabin_karp_all(c[0], c[1])[0], reference_search_all,
           gen_case, n=4000, extra=EDGE, label="rabin_karp")
print("Rabin-Karp with a 2^61-1 modulus: %s cases identical to the reference."
      % "{:,}".format(n))

Rabin-Karp with a 2^61-1 modulus: 4,009 cases identical to the reference.


In [16]:
# ---------------------------------------------------------------------------
# Now shrink the modulus until collisions appear.
# ---------------------------------------------------------------------------
rng = random.Random(11)
haystack = "".join(rng.choice("abcdefgh") for _ in range(20_000))
needle = "hello!"                          # cannot occur: wrong alphabet

print("20,000 characters over the alphabet a-h, searching for %r," % needle)
print("which cannot possibly occur.\n")
print("  %-24s %12s %18s %12s" % ("modulus", "hash hits", "false positives", "verified"))
print("  " + "-" * 70)
for mod in [101, 1009, 100003, (1 << 61) - 1]:
    matches, hits = rabin_karp_all(haystack, needle, mod=mod, verify=True)
    print("  %-24s %12d %18d %12d" % (mod, hits, hits - len(matches), len(matches)))

print()
print("  With verification ON, every modulus gives the right answer: 0 matches.")
print("  The 'false positives' column is the work the verification threw away.")
print()
print("  With verification OFF, those false positives become WRONG ANSWERS:")
for mod in [101, 1009]:
    wrong, _ = rabin_karp_all(haystack, needle, mod=mod, verify=False)
    print("    modulus %-8s reports %3d matches for a pattern that never occurs"
          % (mod, len(wrong)))

print()
print("Two lessons, and the second is the one that bites in production:")
print()
print("  1. The verification step is not optional. It is what makes Rabin-Karp")
print("     CORRECT; the hash only makes it fast.")
print("  2. Verification is also why the worst case is O(nm): an adversary who")
print("     knows your base and modulus can construct a text where every window")
print("     collides, and then you verify at every position. Real systems")
print("     randomise the base per run for exactly this reason -- the same")
print("     defence NB-03 uses for hash tables.")
print()
print("Where Rabin-Karp genuinely wins: searching for MANY patterns at once (hash")
print("them all, one set lookup per window) and 2-D pattern matching. For a single")
print("pattern, KMP is simpler and has no false-positive path at all.")

20,000 characters over the alphabet a-h, searching for 'hello!',
which cannot possibly occur.

  modulus                     hash hits    false positives     verified
  ----------------------------------------------------------------------
  101                               195                195            0
  1009                               20                 20            0
  100003                              0                  0            0
  2305843009213693951                 0                  0            0

  With verification ON, every modulus gives the right answer: 0 matches.
  The 'false positives' column is the work the verification threw away.

  With verification OFF, those false positives become WRONG ANSWERS:
    modulus 101      reports 195 matches for a pattern that never occurs
    modulus 1009     reports  20 matches for a pattern that never occurs

Two lessons, and the second is the one that bites in production:

  1. The verification step is not optional.

***
# Part 3 - The signature difficulty: why naive matching survives

Naive matching is $\Theta(nm)$. KMP is $\Theta(n+m)$. Every textbook presents this as settling the
question — and yet `str.find`, `memmem`, and most standard libraries do **not** use KMP.

They are not wrong. The gap between naive matching's worst case and its typical behaviour is
enormous, and it is worth measuring rather than asserting, because the same reasoning applies
every time you are asked to justify an algorithm choice.

In [17]:
# ---------------------------------------------------------------------------
# Comparisons per character of text, on three kinds of input.
# ---------------------------------------------------------------------------
def naive_count_comparisons(text, pat):
    """Total character comparisons performed scanning the whole text."""
    n, m = len(text), len(pat)
    total = 0
    for i in range(n - m + 1):
        j = 0
        while j < m and text[i + j] == pat[j]:
            total += 1
            j += 1
        if j < m:
            total += 1                     # the comparison that failed
    return total


rng = random.Random(7)
random_text = "".join(rng.choice("ab") for _ in range(100_000))
wordish = " ".join("".join(rng.choice("etaoinshrdlu") for _ in range(rng.randrange(2, 9)))
                   for _ in range(15_000))

CASES = [
    ("random binary text, 8-char pattern", random_text, "abbababb"),
    ("English-like text, 4-char pattern", wordish, "zzzz"),
    ("ADVERSARIAL: a^n text, a^500 b", "a" * 100_000, "a" * 500 + "b"),
]

print("  %-36s %10s %16s %14s" % ("input", "n", "comparisons", "per char"))
print("  " + "-" * 80)
for label, text, pat in CASES:
    c = naive_count_comparisons(text, pat)
    print("  %-36s %10s %16s %14.2f"
          % (label, "{:,}".format(len(text)), "{:,}".format(c), c / len(text)))

print()
print("One to two comparisons per character on realistic text -- and crucially,")
print("that does NOT depend on the pattern length. On the constructed input it")
print("is hundreds, and it grows with the pattern length exactly as O(nm) says.")
print()
print("The reason is simple once stated: on text with any variety, a mismatch")
print("usually happens on the FIRST character of the pattern, so the inner loop")
print("barely runs. Over a 2-letter alphabet that costs ~2 comparisons; over")
print("English text, ~1. The O(nm) bound needs long partial matches that fail")
print("near the END of the pattern, and real text does not contain them.")

  input                                         n      comparisons       per char
  --------------------------------------------------------------------------------
  random binary text, 8-char pattern      100,000          199,579           2.00


  English-like text, 4-char pattern        89,893           89,890           1.00


  ADVERSARIAL: a^n text, a^500 b          100,000       49,849,500         498.50

One to two comparisons per character on realistic text -- and crucially,
that does NOT depend on the pattern length. On the constructed input it
is hundreds, and it grows with the pattern length exactly as O(nm) says.

The reason is simple once stated: on text with any variety, a mismatch
usually happens on the FIRST character of the pattern, so the inner loop
barely runs. Over a 2-letter alphabet that costs ~2 comparisons; over
English text, ~1. The O(nm) bound needs long partial matches that fail
near the END of the pattern, and real text does not contain them.


In [18]:
# ---------------------------------------------------------------------------
# The two regimes as growth curves, not anecdotes.
# ---------------------------------------------------------------------------
def naive_first(case):
    text, pat = case
    n, m = len(text), len(pat)
    for i in range(n - m + 1):
        j = 0
        while j < m and text[i + j] == pat[j]:
            j += 1
        if j == m:
            return i
    return -1


def kmp_first(case):
    r = kmp_search_all(case)
    return r[0] if r else -1


def adversarial(n_):
    """Text a^n, pattern a^(n/10) b -- long partial matches that always fail."""
    return ("a" * n_, "a" * (n_ // 10) + "b")


print("On the adversarial family:\n")
print("  naive:")
growth_table(measure_growth(naive_first, [2_000, 4_000, 8_000, 16_000],
                            setup=adversarial, repeats=3), claim="O(n^2)")
print()
print("  KMP:")
growth_table(measure_growth(kmp_first, [2_000, 4_000, 8_000, 16_000],
                            setup=adversarial, repeats=3), claim="O(n)")

print()
print("Head to head, same input:")
print("  %10s %14s %14s %12s" % ("n", "naive (s)", "KMP (s)", "speed-up"))
print("  " + "-" * 54)
for n_ in [8_000, 16_000, 32_000]:
    case = adversarial(n_)
    tn = measure_growth(naive_first, [1], setup=lambda _x, c=case: c, repeats=3)[0]["seconds"]
    tk = measure_growth(kmp_first, [1], setup=lambda _x, c=case: c, repeats=3)[0]["seconds"]
    print("  %10s %14.4f %14.4f %11.0fx" % ("{:,}".format(n_), tn, tk, tn / tk))

print()
print("The speed-up GROWS with n -- that is the complexity difference, not a")
print("constant factor, and it is why KMP exists.")

On the adversarial family:

  naive:


         n        seconds      ratio
------------------------------------
     2,000       0.036002          -
     4,000       0.150814       4.19
     8,000       0.659266       4.37
    16,000       2.636387       4.00

best fit: O(n^2) (relative error 0.059); next: O(n^3) (1.283)
claimed O(n^2) -> measurement MATCHES the claim

  KMP:
         n        seconds      ratio
------------------------------------
     2,000       0.000450          -
     4,000       0.001031       2.29
     8,000       0.002042       1.98
    16,000       0.004105       2.01

best fit: O(n) (relative error 0.059); next: O(n log n) (0.059)
NOT SEPARABLE: O(n) and O(n log n) fit these timings about equally well.
  Read the ratio column instead, and widen the range of sizes or
  count operations rather than timing them (NB-00 1.7) if you need
  to settle it.
claimed O(n) -> measurement MATCHES the claim

Head to head, same input:
           n      naive (s)        KMP (s)     speed-up
  --------------------

       8,000         0.6174         0.0020         306x


      16,000         2.7073         0.0041         667x


      32,000        10.3118         0.0082        1252x

The speed-up GROWS with n -- that is the complexity difference, not a
constant factor, and it is why KMP exists.


### So which should you use?

The honest answer is the one the standard libraries arrived at:

- **Use the built-in** (`str.find`, `in`, `String.indexOf`). It is written in C, it has been
  tuned for two decades, and for realistic text it beats a hand-written KMP in Python by a wide
  margin *despite* the worse asymptotic bound — the constant factor is doing all the work
  (NB-00 §1.3).
- **Reach for KMP when the input is adversarial or degenerate**: long runs of repeated characters,
  DNA sequences, binary data, or anything a user controls. That is not a hypothetical — it is a
  denial-of-service vector, and it is the same shape of problem as catastrophic regex backtracking.
- **Reach for Rabin-Karp when you have many patterns**, or the problem is 2-D.
- **Know that the real libraries use neither.** CPython's `str.find` uses a hybrid of two-way
  matching and Crochemore-Perrin; glibc's `memmem` uses two-way. These give the linear worst-case
  guarantee *and* the fast typical case, at the cost of being much harder to write.

The transferable point: **an asymptotic bound tells you what an adversary can do to you, not what
your data will do to you.** Both matter, and which one dominates is a question about your inputs
rather than about your algorithm.

***
# Part 4 - Tough questions

***

### Q1. Why are strings immutable, and what does it cost?

<details><summary>Answer</summary>

Immutability buys four things that are hard to give up:

- **Safe sharing.** Pass a string anywhere without a defensive copy; nobody can change it under you.
- **Hashable keys.** A mutable key would break every dictionary containing it (NB-03).
- **A cached hash.** Python computes a string's hash once and stores it in the object.
- **Interning.** Identical literals can share one object — Java does this eagerly, which is why
  §1.4 shows `a == b` being *true* for two separate `"hello"` literals.

It costs exactly one thing, and every performance problem in this notebook follows from it:
**every "modification" allocates a new string and copies.** §1.1 demonstrates this by watching
`id()` change on each `+=`.

| Operation | Cost |
|---|---|
| `s[i]`, `len(s)` | $\Theta(1)$ |
| `s + t` | $\Theta(n+m)$ |
| `s.replace(...)`, `s.upper()` | $\Theta(n)$ |
| `s[i:j]` | $\Theta(j-i)$ — slicing copies |
| `"".join(parts)` | $\Theta(\text{total})$, one allocation |

**The one people miss is slicing.** `s[1:]` is a copy, so a recursive function that calls itself
on `s[1:]` is $\Theta(n^2)$ even though it reads like $\Theta(n)$. Pass an index instead.

**The mutable escape hatches:** `list` of characters, `bytearray` for bytes, `StringBuilder` in
Java, `io.StringIO` when you are writing incrementally.

</details>

***

### Q2. Is `s += chunk` in a loop $O(n^2)$?

<details><summary>Answer</summary>

**The algorithm is. CPython often is not, and that is worse than if it simply were.**

The textbook analysis is right: iteration $i$ copies the $i$ characters accumulated so far, and
$\sum i = \Theta(n^2)$.

But CPython checks whether the left operand's **reference count is 1** — whether that variable is
the only thing pointing at the string. If so, no one can observe a mutation, so it resizes the
buffer in place. §1.2 measures the loop as **exactly $O(n)$** because of this — a fit within 3% of
linear, ratios of 2.0.

**Then it measures two separate cliffs**, and the second is the one that is rarely mentioned.

1. **The reference cliff.** Add one line that keeps a second reference — `keep.append(s)` — and
   the identical loop becomes a clean $O(n^2)$, about **4 seconds against 0.02 at n = 40,000**.
   Two orders of magnitude, from a line that never touches the string being built.
2. **The size cliff.** Change *nothing at all* except how much you build. The in-place resize is a
   `realloc`, and `realloc` can only avoid copying when the memory after the block is free. Once
   the string reaches a few megabytes it is served straight from the OS allocator, where it is
   not. §1.2 measures the very same function as $O(n)$ under ~1 MB (error 0.03) and $O(n^2)$ from
   2 MB to 16 MB (error 0.02). No extra reference involved.

**So the practical answer is: write `"".join(parts)`.** It is $O(n)$ unconditionally, it is faster
than even the optimised `+=`, and it cannot silently regress when someone adds a debug log, a
closure, or an intermediate list.

**And note this does not transfer.** §1.3 measures Java's `String +=` in a loop at roughly **60×
to 330×** slower than `StringBuilder` across n = 10,000 to 40,000, with the ratio growing — the JVM has no equivalent rescue. Java is
slower here and predictable; CPython is faster and has a cliff. Predictable is easier to reason
about.

</details>

***

### Q3. Explain KMP. Why is it linear?

<details><summary>Answer</summary>

Naive matching discards information: when the pattern matches five characters then fails, we
already know what those five characters were. KMP uses that.

**The prefix function** $\pi[i]$ is the length of the longest *proper* prefix of $p[0..i]$ that is
also a suffix of it. For `abacabab`, $\pi = [0,0,1,0,1,2,3,2]$ — §2.2 verifies this against a
brute-force definition over 4,000 random patterns.

**The search.** On a mismatch at pattern position $k$, set $k \leftarrow \pi[k-1]$ rather than
restarting, and **never move the text pointer backwards**.

**Why linear** — and this is the part interviewers actually probe. The text index $i$ advances
exactly $n$ times. The pattern index $k$ increases by at most 1 per step, so it increases at most
$n$ times total; since it never goes below zero, it can therefore *decrease* at most $n$ times in
total across the whole run. The inner `while` is bounded by that total, so the algorithm is
$\Theta(n + m)$. It is an amortised argument, identical in shape to NB-01 §2.3's sliding window.

**Measured:** §3 runs naive and KMP on the same adversarial family. Naive fits $O(n^2)$, KMP fits
$O(n)$, and the speed-up **grows with n** — a few hundred × at n=8,000, roughly double that at
16,000, and past a thousand × at 32,000. The point is not the size of any one of those numbers but
that they keep climbing: a growing ratio is the signature of a complexity difference, where a
constant-factor difference would hold steady.

**The detail to get right:** after a full match, `k = pi[k-1]` continues the scan and finds
*overlapping* occurrences; `k = 0` finds only disjoint ones. Both are legitimate; be explicit
about which you want.

</details>

***

### Q4. What is the Z-algorithm, and when would you prefer it to KMP?

<details><summary>Answer</summary>

$z[i]$ is the length of the longest substring starting at $i$ that is also a prefix of $s$. The
algorithm keeps the rightmost known prefix-match interval — the **Z-box** $[lo, hi)$ — and when $i$
falls inside it, copies the already-computed answer from the mirrored position near the front,
extending by explicit comparison only past $hi$. Since `hi` never decreases, those explicit
comparisons total $O(n)$.

**To search**, run it on `pattern + separator + text` where the separator occurs in neither, and
report every $i$ with $z[i] \ge |pattern|$. §2.3 verifies this against brute force over 5,000
cases.

**Prefer Z when:**

- You want prefix-match lengths at **every** position, not just match locations — that is the
  natural output, and it is what problems about periodicity, borders, and string comparison want.
- You find it easier to get right. Most people do; the Z-box invariant is easier to hold in your
  head than the prefix-function fallback.

**Prefer KMP when:**

- You are **streaming** the text. KMP processes one character at a time and needs only $O(m)$
  state; Z needs the concatenated string in memory, so it is $O(n+m)$ space.
- Space matters for the same reason.

They compute equivalent information — you can derive one from the other — and both are $\Theta(n+m)$.
The choice is about output shape and memory, not speed.

</details>

***

### Q5. How does Rabin-Karp work, and what is its worst case?

<details><summary>Answer</summary>

Compare a **rolling hash** of each window against the pattern's hash. Treating a window as a
base-$b$ number mod $M$, sliding it forward is $O(1)$:

$$ h_{i+1} = \big((h_i - \text{text}[i]\cdot b^{m-1})\cdot b + \text{text}[i+m]\big) \bmod M $$

**Expected $O(n+m)$; worst case $\Theta(nm)$.** The worst case happens when hashes collide
constantly and every hit must be verified character by character.

**The verification step is not optional**, and §2.4 demonstrates why by shrinking the modulus.
Searching a 20,000-character text for a pattern that *cannot occur*:

| modulus | hash hits | false positives |
|---|---|---|
| 101 | 195 | **195** |
| 1009 | 20 | 20 |
| 100,003 | 0 | 0 |
| $2^{61}-1$ | 0 | 0 |

With verification on, every modulus gives the right answer. **With verification off, modulus 101
reports 195 matches for a pattern that never occurs.** The hash makes it fast; the verification
makes it correct.

**The security angle:** an adversary who knows your base and modulus can construct input where
every window collides, forcing $\Theta(nm)$. Real implementations randomise the base per run —
the same defence hash tables use against hash-flooding (NB-03).

**Where it genuinely wins:** searching for **many patterns at once** — hash them all into a set,
one lookup per window, still $O(1)$ per window regardless of how many patterns — and 2-D pattern
matching. For a single pattern, KMP is simpler and has no false-positive path at all.

</details>

***

### Q6. Why do standard libraries not use KMP?

<details><summary>Answer</summary>

Because the worst case they avoid almost never happens, and the constant factor they pay is real.

§3 measures naive matching's comparisons per character of text:

| input | comparisons per character |
|---|---|
| English-like text, 4-char pattern | **1.00** |
| random binary text, 8-char pattern | **2.00** |
| constructed adversarial input | **498** |

On realistic text the inner loop barely runs. Over a two-letter alphabet a mismatch takes two
comparisons on average; over English text it takes one, because a mismatch usually occurs on the
pattern's *first* character. Neither figure depends on the pattern length, which is the whole
point — the $m$ in $O(nm)$ never shows up. The $\Theta(nm)$ bound needs long partial matches that repeatedly
fail near the end, and ordinary text does not contain them.

So a C implementation of naive matching, with no preprocessing and perfect cache behaviour, beats
a Python KMP on realistic input by a wide margin — the constant factor (NB-00 §1.3) dominates
completely.

**What the real libraries actually do** is neither: CPython's `str.find` uses a hybrid of two-way
matching and Crochemore-Perrin, and glibc's `memmem` uses two-way. These give the linear
worst-case guarantee *and* the fast typical case, at the cost of being considerably harder to
write.

**When you should reach for KMP:** input that is degenerate or adversarial — long runs of repeated
characters, DNA, binary data, or anything user-controlled. That last one is a denial-of-service
concern, the same shape as catastrophic regex backtracking.

**The transferable point:** an asymptotic bound tells you what an *adversary* can do to you, not
what your *data* will do to you. Which one matters is a question about your inputs.

</details>

***

### Q7. `len("👨‍👩‍👧")` returns 5. Explain.

<details><summary>Answer</summary>

Because "character" means four different things, and `len()` counts the third.

| Term | What it is |
|---|---|
| **byte** | one octet of some encoding |
| **code unit** | the fixed-size piece an encoding works in — 1 byte for UTF-8, **2 for UTF-16** |
| **code point** | one entry in the Unicode table (`U+1F468`) |
| **grapheme cluster** | what a human calls a character |

**Python's `len()` counts code points. Java's `.length()` counts UTF-16 code units. Neither counts
graphemes.** §1.4 measures the same strings four ways:

| string | `len()` | UTF-8 bytes | UTF-16 units |
|---|---|---|---|
| `"café"` (precomposed) | 4 | 5 | 4 |
| `"café"` (e + combining) | **5** | 6 | 5 |
| G clef `U+1D11E` | 1 | 4 | **2** |
| family emoji | **5** | 18 | 8 |
| flag 🇮🇳 | 2 | 8 | 4 |

The family emoji is one glyph built from three people joined by zero-width joiners — 5 code points.
The flag is two regional-indicator letters. Every number in that table is correct; they answer
different questions.

**The three consequences:**

1. **Truncation splits things.** `s[:10]` can cut a grapheme cluster in half and produce a lone
   combining mark or, in Java, a lone surrogate — which is not valid text.
2. **Equality fails on visually identical strings.** §1.4: precomposed `café` != decomposed
   `café`. macOS stores filenames NFD and most other systems NFC, so a round-trip breaks
   comparison. **Normalise at the boundary** (usually NFC), once.
3. **Database limits are in bytes.** A `VARCHAR(10)` rejects that 18-byte emoji.

**Java-specific:** `charAt()` and `length()` are UTF-16, so iterating with `charAt` splits any
astral character. Use `codePoints()`. For graphemes, both languages need a library — `regex` with
`\X` in Python, `BreakIterator` in Java.

</details>

***

### Q8. Why is `==` on Java strings a bug that passes its tests?

<details><summary>Answer</summary>

Because it compares references, and the compiler makes references coincide exactly in the cases a
test is likely to cover. §1.4 measures it:

```
a == b            true     // two "hello" literals -- interned to one object
a == c            false    // c = new String("hello")
a == c.intern()   true
a.equals(c)       true
```

**Interning** is the JVM keeping a pool of string literals so identical literals share one object.
So `==` works for literals — which is what appears in unit tests — and fails for anything built at
runtime: read from a file, parsed from JSON, concatenated in a loop, or received over a network.
The bug ships.

**Always `.equals()`** (or `Objects.equals` for null-safety).

**The same trap with a different type:** Java caches `Integer` objects for −128..127, so `==` on
boxed integers appears to work and breaks at 128. §1.4 in NB-01 measures this. Small-value caches
are a recurring source of tests that pass for the wrong reason.

**Python's version:** CPython interns short string literals and small integers too, so
`"hello" is "hello"` may be `True` while `("hel" + "lo") is "hello"` is `False`. Use `is` only for
`None`, and for genuine identity checks.

**When interning is useful:** deduplicating many repeated strings to save memory. `String.intern()`
in Java, `sys.intern()` in Python. It is a memory optimisation with a real cost — the pool is
never collected in older JVMs — not something to reach for by default.

</details>

***

### Q9. How would you check whether two strings are anagrams? And rotations?

<details><summary>Answer</summary>

**Anagrams.** Compare character counts.

- Sorting both and comparing: $\Theta(n \log n)$ time, trivial to write, and the right answer in
  an interview unless asked to do better.
- Counting with a hash map or a fixed array: $\Theta(n)$ time, $\Theta(k)$ space for alphabet
  size $k$. Increment for one string, decrement for the other, check all zero.

**The traps:** Unicode (Q7 — normalise first, or `é` and `é` differ), case, whitespace, and
whether the two strings must be the same length (they must; check it first and return early).

**Rotations.** Is `b` a rotation of `a`? The elegant answer:

$$ b \text{ is a rotation of } a \iff |a| = |b| \text{ and } b \text{ is a substring of } a + a $$

because `a + a` contains every rotation of `a` exactly once. `"erbottlewat" in "waterbottle" * 2`
is `True`.

That reduces the problem to substring search, so with KMP (§2.2) it is $\Theta(n)$ and with naive
matching $\Theta(n^2)$ worst case — and this is one of the cases where the worst case is reachable,
since `a = "aaaa...a"` is exactly the degenerate input §3 constructs.

**Do not forget the length check.** Without it, `"a"` counts as a rotation of `"aa"`.

</details>

***

### Q10. Your log-processing job slowed down badly after the input grew. It builds strings. Where do you look?

<details><summary>Answer</summary>

Get the ratio first (NB-00 Q11): ~2 is growth, **~4 is quadratic**. If it is ~4, the string-specific
suspects in order:

1. **`s += chunk` in a loop that lost its optimisation.** §1.2's cliff. Something now holds a
   second reference — a list of intermediate values, a closure, a logging call, a debugger. The
   loop did not change; its refcount did. Fix: `"".join(parts)`.
2. **`s = s + chunk`** (rather than `+=`) never gets the optimisation at all — it is a plain
   binary operation producing a new object.
3. **Slicing in a loop, or recursion on `s[1:]`.** Each slice copies (Q1). Pass indices.
4. **`sub in s` inside a loop** over a large text — $\Theta(nm)$ per call.
5. **Repeated `.replace()` or `.strip()` chains.** Each one is a full pass and a full copy; six
   chained calls are six copies.
6. **Regex backtracking.** A pattern with nested quantifiers can go exponential on input that
   nearly matches. This is a denial-of-service vector, not just slowness.

**If the ratio is ~2 but everything got slower**, suspect encoding: are you decoding and re-encoding
on every operation? Is a `str`/`bytes` conversion happening in the inner loop?

**Then profile** to confirm rather than to explore.

</details>

***

### Q11. When should you use a trie, a suffix structure, or a regex instead?

<details><summary>Answer</summary>

The algorithms in §2 solve *one pattern against one text, once*. Change any of those and a
different structure wins.

| Situation | Use | Why |
|---|---|---|
| **Many patterns, one text** | Aho-Corasick (NB-10), or Rabin-Karp with a hash set | Aho-Corasick is KMP generalised to a trie of patterns: $\Theta(n + \text{total pattern length} + \text{matches})$ |
| **One pattern, many searches of the same text** | suffix array / suffix automaton | Preprocess the *text* once, then each query is $\Theta(m \log n)$ or better |
| **Prefix queries, autocomplete** | trie (NB-10) | $\Theta(\text{key length})$, independent of how many keys are stored |
| **A pattern language, not a fixed string** | regex | Character classes, alternation, repetition |
| **Approximate matching** | edit distance DP (NB-19) | "within 2 edits" is a different problem entirely |

**The regex warning is worth stating explicitly.** A regex is a *language*, not a search — and
backtracking engines (Python's `re`, Java's, PCRE) can go exponential on patterns with nested
quantifiers like `(a+)+b` against `aaaa...a`. That has taken down major sites. If the pattern is a
fixed string, use `str.find`; if it is user-supplied, use a non-backtracking engine (RE2) or bound
the input.

**And the one people over-reach for:** a suffix array costs $\Theta(n \log n)$ to build and
$\Theta(n)$ memory with a large constant. For a single search of a text you will not search again,
that preprocessing is pure loss.

</details>

***

### Q12. What is a rolling hash good for beyond string search?

<details><summary>Answer</summary>

Anything where you need to compare **many overlapping windows** cheaply. The rolling property —
$O(1)$ update as the window slides — is the whole value, and it generalises well beyond §2.4.

- **Deduplication and delta compression.** `rsync` and content-defined chunking split files at
  positions where a rolling hash has a particular form, so an insertion near the start shifts only
  one chunk boundary rather than all of them. This is why `rsync` transfers little after a small
  edit.
- **Plagiarism and clone detection.** Hash every $k$-gram of two documents and compare the sets;
  winnowing keeps a representative subset.
- **Bioinformatics.** Minimisers over $k$-mers, the same idea, for read alignment.
- **2-D pattern matching.** Hash each row of the pattern, then run a rolling hash down the columns
  of row-hashes.
- **Longest common substring, by binary search on the length.** For a candidate length $L$, hash
  every $L$-window of both strings and intersect. That gives $\Theta(n \log n)$ expected, and it is
  a genuinely useful competitive-programming technique.
- **Detecting duplicate substrings** within one string, same way.

**Two cautions carried over from §2.4.** First, **verify**, or accept a probabilistic answer
knowingly — for deduplication a collision means data corruption, so those systems use cryptographic
hashes for the final check. Second, **randomise the base** if inputs are adversarial; a fixed base
and modulus can be attacked, and "anti-hash tests" defeating fixed parameters are a known sport in
competitive programming.

</details>

***

## Coding challenges

### Challenge 1 — build Aho-Corasick

§2 searched for one pattern. Q11 says many patterns want a different algorithm; build it.

1. Build a trie of the patterns (NB-10 previews the structure).
2. Add **failure links** — the generalisation of KMP's prefix function to a trie: on a mismatch,
   follow the link to the longest proper suffix that is a node in the trie. Build them with a BFS.
3. Search a text once, reporting every occurrence of every pattern. Verify against running
   §2.2's KMP separately per pattern.
4. Measure both against pattern-count: KMP-per-pattern should be linear in the number of patterns,
   Aho-Corasick roughly flat. That crossover is the whole argument for it.

***

### Challenge 2 — break Rabin-Karp on purpose

§2.4 showed collisions appearing when the modulus is small. Now attack a realistic one.

1. Fix base 256 and a 32-bit modulus. Construct two different strings of the same length with the
   same hash. (Hint: the hash is linear, so look for a difference vector that sums to zero mod M —
   birthday search over $2^{16}$ random strings will find one.)
2. Build a text where a large fraction of windows collide with the pattern, and measure the search
   degrading toward $\Theta(nm)$.
3. Now randomise the base per run and show the attack fails.
4. Write the two-sentence rule you would put in a code review about rolling hashes on
   user-controlled input.

***

### Challenge 3 — the grapheme-correct string library

Q7 established that neither language counts what users call characters. Fix it for a small API.

1. Implement `grapheme_len`, `grapheme_slice` and `grapheme_reverse` using Unicode grapheme
   cluster boundaries (`regex` module's `\X`, or implement UAX #29's rules for the common cases).
2. Test them on: combining marks, emoji ZWJ sequences, flags, and Devanagari or Thai text where a
   cluster spans several code points.
3. Show `s[::-1]` producing broken output where your `grapheme_reverse` does not.
4. Measure the cost. Grapheme segmentation is much slower than `len()` — quantify it, and write
   down where in a real system you would and would not pay it.

***
# Part 5 - Practice

| # | Exercise | Skill it forces | Difficulty |
|---|---|---|---|
| 1 | Reverse words in a sentence | Immutability and in-place thinking | ★☆☆☆☆ |
| 2 | Palindrome, done properly | Unicode, case, and what "same" means | ★★☆☆☆ |
| 3 | Longest common prefix | Where the naive answer is already optimal | ★★☆☆☆ |
| 4 | Implement `str.find` and beat it | The constant factor from §3 | ★★★☆☆ |
| 5 | Longest palindromic substring | Manacher's, and why it is Z in disguise | ★★★★☆ |
| 6 | Minimum window substring | Sliding window with a frequency map | ★★★★☆ |
| 7 | String periodicity | Reading the prefix function for structure | ★★★☆☆ |
| 8 | Build a diff | Edit distance made practical | ★★★★☆ |

***

### 1. Reverse the words in a sentence

`"the sky is blue"` → `"blue is sky the"`.

1. Do it with `split` and `join`, then count how many intermediate strings that allocates.
2. Now do it on a `list` of characters with $O(1)$ extra space: reverse the whole thing, then
   reverse each word. Verify against the simple version.
3. Handle leading, trailing and repeated spaces. Stress-test — this is where the bugs are.
4. In Java, do it on a `char[]`. Compare with the `String.split` version and explain the gap.

***

### 2. Palindrome, done properly

1. Start with the naive `s == s[::-1]`.
2. Now handle: case-insensitivity, ignoring punctuation, and **Unicode**. Is `"Ésé"` a palindrome?
   What about the decomposed form? Normalise first (§1.4) and say which form and why.
3. Do it with two pointers and $O(1)$ space (NB-01 §2.3).
4. Find a string that your implementation calls a palindrome and a human would not, or vice versa.
   There is at least one involving combining marks.

***

### 3. Longest common prefix of a list of strings

1. Implement it. Prove the complexity is $\Theta(\text{total input size})$ in the worst case and
   that no algorithm can do better — you must read the common prefix.
2. Compare horizontal scanning, vertical scanning, and divide-and-conquer. They have the same
   bound; measure the constants.
3. Now the trie version (NB-10). It is worse for one query and better for many — find the number
   of queries at which it wins.

***

### 4. Implement `str.find`, then try to beat the built-in

1. Implement naive, KMP and Rabin-Karp with a common interface, all verified against
   `str.find`.
2. Benchmark all four on: English text, random binary, and §3's adversarial input.
3. You will lose to the built-in on the first two by a wide margin. Explain why using NB-00 §1.3,
   then find the input size and shape where your KMP finally wins.
4. Read CPython's `stringlib/fastsearch.h` and identify which algorithm it actually uses.

***

### 5. Longest palindromic substring

1. Start with expand-around-centre, $\Theta(n^2)$. Verify against brute force.
2. Implement **Manacher's algorithm** for $\Theta(n)$. Then look at its structure and explain how
   the "mirror" reuse is the same idea as §2.3's Z-box.
3. Handle even-length palindromes with the sentinel trick, and say why it works.
4. Measure both on a string of all-equal characters — the worst case for the quadratic version.

***

### 6. Minimum window substring

Smallest window of `s` containing all characters of `t`, with multiplicity.

1. Sliding window plus a frequency map (NB-01 §2.3). State the invariant before you code it.
2. Prove it is $\Theta(|s| + |t|)$: each pointer advances at most $|s|$ times.
3. Verify against a brute-force $\Theta(n^2)$ reference over thousands of random cases.
4. Then break it: find the input where a naive "shrink while valid" gets the wrong answer, and say
   which part of the invariant it violated.

***

### 7. String periodicity from the prefix function

1. Prove: a string $s$ of length $n$ has period $n - \pi[n-1]$, and it is the smallest period.
2. Use it to answer "is `s` a whole number of repetitions of some shorter string?" in $\Theta(n)$.
3. Verify against a brute-force check over all divisors.
4. Use it to find the smallest number of characters to append to make `s` a repetition.

***

### 8. Build a diff

1. Implement edit distance with the DP table (NB-19 previews the technique).
2. Recover the actual edit *script*, not just the distance — that means backtracking through the
   table, and it is the part people skip.
3. Reduce space to $\Theta(\min(n,m))$ for the distance. Then explain why recovering the script
   from the reduced version needs Hirschberg's algorithm.
4. Compare your output to `difflib` on two versions of a real file, and explain any differences —
   `difflib` optimises for human readability rather than minimum edits.

***
# Part 6 - Reading

## Start here

**1. *Introduction to Algorithms* (CLRS), chapter 32 — "String Matching".**
> Naive, Rabin-Karp, finite automata and KMP, with the amortised proof of KMP's linearity done
> properly. §2.2's argument is this chapter compressed; read it for the version with the potential
> function written out.

**2. [Fast Pattern Matching in Strings](https://doi.org/10.1137/0206024)** —
Knuth, Morris & Pratt, SIAM J. Computing 6(2), 1977.
> The original. Worth reading for how much of it is about the *automaton* view rather than the
> prefix-function view most courses teach, and for Knuth's account of deriving it from Cook's
> theorem on two-way pushdown automata — a rare glimpse of where an algorithm came from.

**3. [The Absolute Minimum Every Software Developer Absolutely, Positively Must Know About Unicode and Character Sets](https://www.joelonsoftware.com/2003/10/08/the-absolute-minimum-every-software-developer-absolutely-positively-must-know-about-unicode-and-character-sets-no-excuses/)** —
Joel Spolsky, 2003. **Free.**
> Twenty minutes, and it is the reason §1.4 exists. Dated in its examples and exactly right in its
> central claim: there is no such thing as plain text, and code that assumes otherwise is broken
> in a way you will not notice until a user with the wrong name arrives.

## The source behind each section

| Section | Where it comes from | Free? |
|---|---|---|
| 1.1–1.2 — immutability, the `+=` optimisation | **CPython** `Objects/unicodeobject.c`, `unicode_concatenate` — the refcount check is right there in the source | ✅ |
| 1.3 — builders | **JLS** §15.18.1 on string concatenation; the JDK's `StringBuilder` | ✅ |
| 1.4 — **encodings and graphemes** | **Unicode Standard Annex #29**, *Text Segmentation* — [unicode.org/reports/tr29](https://unicode.org/reports/tr29/); **UAX #15** on normalisation | ✅ |
| 1.4 — the practical version | **Spolsky**, 2003; and **McIlroy**, *Unicode in practice* | ✅ |
| 2.2 — **KMP** | **Knuth, Morris & Pratt**, SIAM J. Comp. **1977**; **CLRS** ch. 32.4 | 🔍 |
| 2.3 — Z-algorithm | **Gusfield**, *Algorithms on Strings, Trees and Sequences*, ch. 1 — where the Z-box presentation originates | 🔍 |
| 2.4 — **Rabin-Karp** | **Karp & Rabin**, *Efficient randomized pattern-matching algorithms*, IBM J. Res. Dev. **1987** | 🔍 |
| 3 — what libraries actually use | **Crochemore & Perrin**, *Two-way string-matching*, JACM **1991**; CPython's `stringlib/fastsearch.h` | ✅ (the source) |
| Q11 — Aho-Corasick | **Aho & Corasick**, *Efficient string matching: an aid to bibliographic search*, CACM **1975** | 🔍 |
| Q11 — regex blowup | **Cox**, *Regular Expression Matching Can Be Simple And Fast* — [swtch.com/~rsc/regexp/regexp1.html](https://swtch.com/~rsc/regexp/regexp1.html) | ✅ |
| Q12 — rolling hashes in practice | **Tridgell**, the rsync algorithm — [rsync.samba.org/tech_report](https://rsync.samba.org/tech_report/) | ✅ |

**Legend:** ✅ free at the link · 🔍 search the exact title on
[Google Scholar](https://scholar.google.com)

### If you read only one

**Cox, *Regular Expression Matching Can Be Simple And Fast*.** It is not about the algorithms in
§2, and that is the point: it takes the same tension this notebook keeps running into — an
algorithm with a good typical case and a catastrophic worst case, deployed where users control the
input — and follows it to its conclusion, which is that Perl, Python, Java and PCRE all made the
same wrong choice and Thompson made the right one in 1968.

Read it against §3, which measures naive matching being fine on real text and 500× worse on
constructed input. Cox's version of that observation is the one with the security implications, and
it will change how you review a regex.

Then read **Spolsky** for §1.4, and **CLRS ch. 32** for the KMP proof.

***
# Appendix

| Symptom | Cause | Fix |
|---|---|---|
| String building suddenly 100× slower | CPython's in-place `+=` lost its refcount-1 fast path — something took a second reference (§1.2) | `"".join(parts)` — unconditional $O(n)$ |
| String building fine in tests, quadratic in production | The `+=` fast path also dies on **size**: past a few MB `realloc` must copy (§1.2) | `"".join(parts)`; test at production sizes |
| `s = s + chunk` in a loop is slow | Never gets the optimisation at all (§1.2) | `+=` at best, `join` properly |
| Java string building is slow | `String +=` allocates per iteration (§1.3) | `StringBuilder` |
| Recursion on `s[1:]` is quadratic | Slicing copies (Q1) | Pass an index |
| `len()` disagrees with what users see | `len()` counts code points, not graphemes (§1.4) | Grapheme segmentation; `regex` `\X` or `BreakIterator` |
| Java `length()` is 2 for one emoji | Java strings are UTF-16 (§1.4) | `codePointCount`, `codePoints()` |
| Identical-looking strings compare unequal | Precomposed vs decomposed (§1.4) | `unicodedata.normalize("NFC", s)` at the boundary |
| `VARCHAR(n)` rejects valid input | Column limits are bytes, `len()` is code points (§1.4) | Measure `len(s.encode("utf-8"))` |
| Java `==` works in tests, fails in production | Literals are interned; runtime strings are not (Q8) | `.equals()`, always |
| Truncating text produces mojibake | Cut mid-cluster or mid-surrogate (§1.4) | Truncate on grapheme boundaries |
| Rabin-Karp reports matches that are not there | Verification step omitted (§2.4) | Compare the actual substring on a hash hit |
| Rabin-Karp degrades to $O(nm)$ | Adversarial input colliding with a fixed base (§2.4) | Randomise the base per run |
| Substring search hangs on odd input | Naive matching's worst case is reachable (§3) | KMP, or the built-in |
| Regex hangs on a long input | Catastrophic backtracking (Q11) | Rewrite the pattern; RE2; bound the input |
| KMP finds too few matches | `k = 0` after a match instead of `k = pi[k-1]` (§2.2) | Decide whether overlaps count, and say so |

## Checklist for string code

- [ ] Is any string built in a loop with `+=` rather than `join` (§1.2)?
- [ ] Was that loop measured at **production sizes**, not test sizes (§1.2's size cliff)?
- [ ] Any slicing or `s[1:]` recursion inside a loop (Q1)?
- [ ] Is text **normalised** (usually NFC) where it enters the system (§1.4)?
- [ ] Are length limits measured in the same unit the storage uses — bytes vs code points (§1.4)?
- [ ] Does any truncation respect grapheme boundaries (§1.4)?
- [ ] In Java, is `.equals()` used rather than `==` (Q8)?
- [ ] In Java, does any iteration use `charAt` where it should use `codePoints` (§1.4)?
- [ ] If a rolling hash is used, is there a **verification** step (§2.4)?
- [ ] If input is user-controlled, is the search worst case bounded (§3, Q11)?
- [ ] Is a regex being used where a fixed-string `find` would do (Q11)?
- [ ] Was the complexity **measured**, not assumed (NB-00 §1.4)?

## Where to go next

| Notebook | Why it follows |
|---|---|
| `hashing_zero_to_hero.ipynb` | §2.4's rolling hash and its adversarial worst case, as a whole topic |
| `tries_zero_to_hero.ipynb` | Q11's answer for many patterns and prefix queries, and Aho-Corasick |
| [`arrays_zero_to_hero.ipynb`](arrays_zero_to_hero.ipynb) | The contiguity §1.1 builds on, and the quadratic accident §1.2 is a special case of |
| `dynamic_programming_zero_to_hero.ipynb` | Edit distance and LCS — approximate matching, where §2's exact algorithms stop applying |

See [`README.md`](README.md) for the full roster and reading order.